In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as scipy_stats
import gpboost as gpb
import patsy
import re

In [ ]:
results_path = 'results_0'
formats = ['parquet', 'delta', 'hudi', 'iceberg']
data_sets = ['tpcds_1', 'tpcds_10', 'tpcds_100']
optimizations = ['none', 'zorder', 'bloom', 'partitioning']
block_sizes = ['64MiB', '128MiB', '256MiB']
data_frames = []

In [ ]:
for format in formats:
    for data_set in data_sets:
        for optimization in optimizations:
            for block_size in block_sizes:
                if format == 'parquet' and (optimization != 'none' or block_size != '128MiB'):
                    continue
                
                file_path = f'{results_path}/{format}/{data_set}/{optimization}/{block_size}/results.csv'
                df = pd.read_csv(file_path)
                df['format'] = format
                df['data_set'] = data_set
                df['optimization'] = optimization
                df['block_size'] = block_size
                data_frames.append(df)

In [ ]:
master_df = pd.concat(data_frames, ignore_index=True)
target_metric = 'elapsedTime'
master_df[target_metric] = master_df[target_metric].round(2)
summary_stats_by_query = master_df.groupby(['format', 'data_set', 'optimization', 'block_size', 'query'])[target_metric].agg(
    mean='mean',
    median='median',
    std='std',
    min='min',
    max='max',
    cv=lambda x: np.std(x, ddof=1) / np.mean(x) * 100 # coef of variation
).reset_index()
display(summary_stats_by_query)

In [ ]:
print("--- Shapiro-Wilk Test for Normality ---")
groups_shapiro = master_df.groupby(['format', 'data_set', 'optimization', 'block_size', 'query'])[target_metric].apply(list)

df_shapiro_list = []

for name, group_data in groups_shapiro.items():
    stat, p_val = scipy_stats.shapiro(group_data)

    df_shapiro_list.append({
        'Group': name,
        'W_Statistic': stat,
        'p_value': p_val.round(4),
        'Distribution': 'Normal' if p_val > 0.05 else 'Not Normal'
    })

df_shapiro = pd.DataFrame(df_shapiro_list)
display(df_shapiro)

df_shapiro_normal = df_shapiro[df_shapiro['Distribution'] == 'Normal']
display(df_shapiro_normal)

df_shapiro_not_normal = df_shapiro[df_shapiro['Distribution'] == 'Not Normal']
display(df_shapiro_not_normal)

In [ ]:
master_df = pd.concat(data_frames, ignore_index=True)
summary_stats_aggregated_runs = master_df.groupby(['format', 'data_set', 'optimization', 'block_size', 'run_id'])[['elapsedTime']].sum().reset_index()
display(summary_stats_aggregated_runs)

In [ ]:
target_metrics = ['elapsedTime', 'executorCpuTime', 'executorRunTime', 'resultSize', 'peakExecutionMemory', 'shuffleTotalBytesRead', 'shuffleBytesWritten']
EPSILON = 1e-6

In [ ]:
def print_df_latex(df, caption):
    latex_output = (
        df.style
        .hide(axis='index')
        .to_latex(
            caption=caption, 
            position='h!', 
            position_float='centering',
            hrules=True
        )
    )
    lines = latex_output.split('\n')
    caption_line = next((l for l in lines if l.strip().startswith('\\caption{')), None)
    
    if caption_line:
        lines.remove(caption_line)
        # Find the index of \end{table} and insert the caption before it
        end_table_idx = next(i for i, l in enumerate(lines) if l.strip() == '\\end{table}')
        lines.insert(end_table_idx, caption_line)
        latex_output = '\n'.join(lines)

    print(latex_output)

def clean_patsy_name(name):
    if name == 'Intercept':
        return name
    cleaned = re.sub(r'C\([^,)]+,\s*Treatment\([^)]+\)\)', '', name)
    cleaned = re.sub(r'\[T\.([^\]]+)\]', r' \1', cleaned)
    return cleaned.title().strip()

In [ ]:
for target_metric in target_metrics:
        
    for data_set in data_sets:

        df_rq1 = master_df[
            (master_df['optimization'] == 'none') &
            (master_df['block_size'] == '128MiB') &
            (master_df['data_set'] == data_set) 
        ].copy()

        df_rq1['format'] = pd.Categorical(df_rq1['format'], categories=formats)

        y = df_rq1[target_metric].values + EPSILON
        group_data = df_rq1['query'].to_numpy()

        X_df = patsy.dmatrix('format', data=df_rq1, return_type='dataframe')
        X = X_df.values
        feature_names = X_df.columns.tolist()

        gp_model = gpb.GPModel(group_data=group_data, likelihood='gamma')
        gp_model.fit(y=y, X=X)
        fixed_effects, std_errs = gp_model.get_coef(format_pandas=False, std_err=True)

        summary_data = []
        for name, coef, std_err in zip(feature_names, fixed_effects, std_errs):
            z_stat = coef / std_err
            p_value = 2 * scipy_stats.norm.sf(np.abs(z_stat))
            pct_change = (np.exp(coef) - 1) * 100 if name != 'Intercept' else 0
            
            summary_data.append({
                'Factor': 'Format Parquet' if name == 'Intercept' else clean_patsy_name(name),
                'Coefficient $\\beta$': f'{coef:.4f}',
                'Std. Error': std_err  if name != 'Intercept' else '',
                'Percentage Change': f'{pct_change:+.1f}\\%',
                '$p$-Value': f'{p_value:.4f}' if name != 'Intercept' else '',
                '$H_0$ Rejected': str(p_value <= 0.05) if name != 'Intercept' else ''
            })

        summary_df = pd.DataFrame(summary_data)
        caption_text = f"TPC-DS {data_set.split('_')[1]}GB, {re.sub(r'(?<!^)(?=[A-Z])', ' ', target_metric).title()}"
        print_df_latex(summary_df, caption_text)

In [ ]:
for format in ['delta', 'hudi', 'iceberg']:

    for target_metric in target_metrics:
        
        for data_set in data_sets:

            df_rq2 = master_df[
                (master_df['format'] == format) &
                (master_df['data_set'] == data_set) 
            ].copy()

            df_rq2['format'] = pd.Categorical(df_rq2['format'], categories=formats)

            y = df_rq2[target_metric].values + EPSILON
            group_data = df_rq2['query'].to_numpy()

            X_df = patsy.dmatrix("C(optimization, Treatment(reference='none')) * C(block_size, Treatment(reference='128MiB'))", data=df_rq2, return_type='dataframe')
            X = X_df.values
            feature_names = X_df.columns.tolist()

            gp_model = gpb.GPModel(group_data=group_data, likelihood='gamma')
            gp_model.fit(y=y, X=X)
            fixed_effects, std_errs = gp_model.get_coef(format_pandas=False, std_err=True)

            summary_data = []
            for name, coef, std_err in zip(feature_names, fixed_effects, std_errs):
                z_stat = coef / std_err
                p_value = 2 * scipy_stats.norm.sf(np.abs(z_stat))
                pct_change = (np.exp(coef) - 1) * 100 if name != 'Intercept' else 0
                
                summary_data.append({
                    'Factor':  'No opt: 128Mib' if name == 'Intercept' else clean_patsy_name(name),
                    'Coefficient $\\beta$': f'{coef:.4f}',
                    'Std. Error': f'{std_err:.4f}' if name != 'Intercept' else '',
                    'Percentage Change': f'{pct_change:+.1f}\\%' if not np.isnan(pct_change) else 'Baseline',
                    '$p$-Value': f'{p_value:.4f}' if name != 'Intercept' else '',
                    '$H_0$ Rejected': str(p_value <= 0.05) if name != 'Intercept' else ''
                })

            summary_df = pd.DataFrame(summary_data)
            caption_text = f'Format {format.capitalize()}, TPC-DS {data_set.split('_')[1]}GB, {re.sub(r'(?<!^)(?=[A-Z])', ' ', target_metric).title()}'
            print_df_latex(summary_df, caption_text)
            

In [ ]:
for target_metric in target_metrics:
        
    for data_set in data_sets:

        for optimization in optimizations:

            df_rq3 = master_df[
                ((master_df['format'] == 'parquet') & (master_df['data_set'] == data_set)) |
                ((master_df['format'] != 'parquet') & (master_df['data_set'] == data_set) & (master_df['optimization'] == optimization))
            ].copy()

            # df_rq3['format'] = pd.Categorical(df_rq3['format'], categories=formats)
            df_rq3['format_variation'] = df_rq3['format'] + '_' + df_rq3['optimization'] + '_' + df_rq3['block_size'].astype(str)

            y = df_rq3[target_metric].values + EPSILON
            group_data = df_rq3['query'].to_numpy()

            X_df = patsy.dmatrix("C(format_variation, Treatment(reference='parquet_none_128MiB'))", data=df_rq3, return_type='dataframe')
            X = X_df.values
            feature_names = X_df.columns.tolist()

            gp_model = gpb.GPModel(group_data=group_data, likelihood='gamma')
            gp_model.fit(y=y, X=X)
            fixed_effects, std_errs = gp_model.get_coef(format_pandas=False, std_err=True)

            summary_data = []
            for name, coef, std_err in zip(feature_names, fixed_effects, std_errs):
                z_stat = coef / std_err
                p_value = 2 * scipy_stats.norm.sf(np.abs(z_stat))
                pct_change = (np.exp(coef) - 1) * 100 if name != 'Intercept' else 0
                
                summary_data.append({
                    'Factor': 'Parquet None 128MiB' if name == 'Intercept' else clean_patsy_name(name).replace('_', ' '),
                    'Coef. $\\beta$': f'{coef:.4f}',
                    'Std. Error': f'{std_err:.4f}' if name != 'Intercept' else '',
                    'Pct. Change': f'{pct_change:+.1f}\\%',
                    '$p$-Value': f'{p_value:.4f}' if name != 'Intercept' else '',
                    '$H_0$ Rejected': str(p_value <= 0.05) if name != 'Intercept' else ''
                })

            summary_df = pd.DataFrame(summary_data)
            caption_text=f'Optimization {optimization.capitalize()}, TPC-DS {data_set.split('_')[1]}GB, {re.sub(r'(?<!^)(?=[A-Z])', ' ', target_metric).title()}'
            print_df_latex(summary_df, caption_text)